# Custom bars with `agg` dictionaries

`resample` reduces a stream into bars. Its `agg` argument accepts string shorthands
(`"first"`, `"last"`, `"ohlc"`, `"ohlcv"`, ...) for common cases; for full control,
pass a dict whose values are lazy expressions - one per column, all sharing a single bar
clock. The expressions are assembled in a `Dag` and bound to data at call time, so the
same graph runs batch or live with identical results.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from screamer import First, Last, ExpandingMax, ExpandingMin, ExpandingSum, PosPart, NegPart
from screamer.streams import resample
from screamer.dag import Input, Dag

rng = np.random.default_rng(7)
n = 400
price = 100 + np.cumsum(rng.normal(size=n))
signed_vol = rng.normal(size=n) * rng.integers(1, 5, size=n)   # buys positive, sells negative
t = np.arange(n, dtype=np.int64)
BAR = 40


## OHLC bars with buy and sell volume

Open, high, low, and close come from the price stream via `First`, `ExpandingMax`,
`ExpandingMin`, and `Last`. Buy and sell volume come from the signed volume stream:
each tick's positive part accumulates into `buy`, its magnitude when negative into
`sell`, via `ExpandingSum` over `PosPart` and `NegPart`. All six columns share one
bar clock and cannot drift relative to each other.


In [ ]:
# Save data arrays; the names below will be reused for graph Inputs.
price_arr, vol_arr, t_arr = price, signed_vol, t

price, vol, t = Input("price"), Input("vol"), Input("t")
bars = resample(t, every=BAR, agg={
    "open":  First()(price),
    "high":  ExpandingMax()(price),
    "low":   ExpandingMin()(price),
    "close": Last()(price),
    "buy":   ExpandingSum()(PosPart()(vol)),
    "sell":  ExpandingSum()(NegPart()(vol)),
})
dag = Dag([t, price, vol], [bars])
ohlcv2 = dag(t=(t_arr.astype(float), t_arr), price=(price_arr, t_arr), vol=(vol_arr, t_arr))

print("columns:", ohlcv2.columns)
o, h, l, c = ohlcv2["open"], ohlcv2["high"], ohlcv2["low"], ohlcv2["close"]
buy, sell = ohlcv2["buy"], ohlcv2["sell"]


In [ ]:
fig = plt.figure(figsize=(10, 6))
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

x = np.arange(len(o))
for i in range(len(x)):
    color = "green" if c[i] >= o[i] else "red"
    ax1.plot([x[i], x[i]], [l[i], h[i]], color=color, lw=1)
    ax1.plot([x[i], x[i]], [o[i], c[i]], color=color, lw=6)
ax1.set_ylabel("price")
ax1.set_title("OHLC bars with buy and sell volume")

ax2.bar(x, buy, color="green", width=0.6)
ax2.bar(x, -sell, color="red", width=0.6)
ax2.axhline(0, color="k", lw=0.5)
ax2.set_ylabel("volume")
ax2.set_xlabel("bar")
plt.tight_layout()


## Time-based and tick-count bucketing

`every=BAR` buckets by time: each bar spans `BAR` index units regardless of tick count.
`count=N` buckets by event count instead, closing a bar after every N ticks. The `agg`
dict and column alignment are identical in both cases.

### Clock-driven empty bars

With sparse data, a time bar may have no trades at all. Pass a dense clock stream
alongside the trade data: each clock tick that crosses a bar boundary finalizes the
current bar even with no events inside it. Use `fill="nan"` to emit all-NaN rows for
empty bars.


In [ ]:
MINUTE = 60

# Six trades across four minutes; minute 2 (t = 120-179) has no activity.
trade_t = np.array([10., 40., 65., 90., 195., 230.])
trade_p = np.array([100., 101., 99., 102., 105., 104.])
trade_v = np.array([8., -3., 6., -5., 10., -4.])

# Dense clock: one tick per minute up to t = 240.
clock_t = np.arange(0., 241., MINUTE)

# Merge trades with clock ticks; price and vol are NaN at clock-only timestamps.
merged_t = np.concatenate([trade_t, clock_t])
order     = np.argsort(merged_t, kind="stable")
merged_t  = merged_t[order]
merged_p  = np.concatenate([trade_p, np.full(len(clock_t), np.nan)])[order]
merged_v  = np.concatenate([trade_v, np.full(len(clock_t), np.nan)])[order]

p_in, v_in, t_in = Input("price"), Input("vol"), Input("t")
min_bars = resample(t_in, every=MINUTE, fill="nan", agg={
    "open":  First()(p_in),
    "high":  ExpandingMax()(p_in),
    "low":   ExpandingMin()(p_in),
    "close": Last()(p_in),
    "buy":   ExpandingSum()(PosPart()(v_in)),
    "sell":  ExpandingSum()(NegPart()(v_in)),
})
min_dag = Dag([t_in, p_in, v_in], [min_bars])
minute_bars = min_dag(
    t=(merged_t, merged_t),
    price=(merged_p, merged_t),
    vol=(merged_v, merged_t),
)

pd.DataFrame(
    minute_bars.values.round(2),
    index=pd.Index(minute_bars.index.astype(int), name="bar_start"),
    columns=list(minute_bars.columns),
)


The bar at `bar_start=120` is all-NaN: the clock tick at t=180 crossed the boundary and
forced finalization of an empty window. The `agg` dict is open-ended - add a column such
as `ExpandingStd()(price)` for per-bar realized volatility; one clock keeps all columns
aligned. The same `Dag` runs batch or live with identical results.
